In [19]:
import pandas as pd
import matplotlib.pyplot as plt 
import matplotlib.ticker as mticker 
from matplotlib.backends.backend_pdf import PdfPages

s = pd.read_csv('sales_transactions_cleaned.csv')
p = pd.read_csv('products.csv')
s['revenue'] =  (s['quantity']*s['price'])- pd.to_numeric(s['discount_amount'],errors='coerce').fillna(0)

In [20]:
def clean_num(col):
    return pd.to_numeric(col.astype(str).str.replace(r'[^0-9.\-]','', regex=True),errors='coerce').abs()

p['price_c'] = clean_num(p['price'])
p['cost_c'] = clean_num(p['cost'])

p['category'] = p['category'].replace({'Pastry':'Pastries'})

p['profit_margin'] = p['price_c'] - p['cost_c']


In [21]:
q = (s.assign(revenue=(s['quantity'] * s['price']) - s['discount_amount'].fillna(0)).merge(p[['product_id','product_name','category','profit_margin']],on='product_id', how='left'))

c = (q[q['category'].isin(['Pastries','Bread','Tarte'])].groupby('category')['revenue'].sum().reindex(['Pastries','Bread','Tarte']))

t = (q.groupby('product_name')[['quantity','revenue']].sum().nlargest(3, 'quantity').reset_index().rename(columns={'product_name' : 'Product Name','quantity' : 'Total Quantity', 'revenue' : 'Total Revenue'}))
t['Total Revenue'] = t['Total Revenue'].apply(lambda v: f'${v:,.2f}')

print(c)
print(t)

category
Pastries    55469.37
Bread          45.81
Tarte       61328.50
Name: revenue, dtype: float64
        Product Name  Total Quantity Total Revenue
0  cannelé bordelais           11965    $55,457.02
1   pain de campagne            7064    $32,487.58
2   macaron pistache            6801    $31,045.62


In [22]:
with PdfPages('Session1_ProductPerformanceVTest.pdf') as pdf:
    
    fig, ax = plt.subplots(figsize=(9,5))
    c.plot(kind='bar', color=['salmon','skyblue','seagreen'], ax=ax, rot=0)
    ax.set_title('Total Revenue by Product Category', fontsize=14, fontweight='bold', pad=12)
    ax.set_ylabel('Total Revenue ($)'); ax.set_xlabel('Categoty')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.2f}'))
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches='tight'); plt.close()

    fig, ax = plt.subplots(figsize=(9,2.5))
    ax.axis('off')
    ax.set_title('Top 3 Best-selling Products',fontsize=14, fontweight='bold', pad=12)
    tbl = ax.table(cellText=t.values, colLabels=t.columns,loc='center',cellLoc='center')
    tbl.set_fontsize(12); tbl.scale(1.5,2.2)
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches='tight'); plt.close()

print('Success')

Success
